In [1]:
import json
import os
import pandas as pd
import janitor  # noqa: F401

JAN_DIR   = "../data/blacklight_json"
APR_DIR   = "../data/blacklight_json_2026"
TARGETS   = "../data/rescan_targets_2026.csv"
YG_DOMAIN = "../data/yg/yg_ind_domain.csv"

### Tracker prevalence: Jan 2025 vs. Apr 2026

In [2]:
def indicators(path):
    with open(path) as f:
        d = json.load(f)
    groups = d.get('groups') or []
    if not groups:
        return {}
    return {c['cardType']: bool(c.get('testEventsFound'))
            for c in groups[0].get('cards', [])
            if c.get('cardType')}

In [3]:
targets = pd.read_csv(TARGETS).clean_names()

rows = []
for domain in targets['private_domain']:
    fname = domain.replace('.', '_') + '.json'
    apr_path = os.path.join(APR_DIR, fname)
    if not os.path.exists(apr_path):
        continue
    jan_path = os.path.join(JAN_DIR, fname)
    row = {'domain': domain}
    apr = indicators(apr_path)
    jan = indicators(jan_path) if os.path.exists(jan_path) else {}
    for ct in set(jan) | set(apr):
        row[f'jan_{ct}'] = jan.get(ct)
        row[f'apr_{ct}'] = apr.get(ct)
    rows.append(row)

df = pd.DataFrame(rows)
df.shape

(500, 19)

In [4]:
CATEGORIES = [
    ('ddg_join_ads',          'Ad trackers'),
    ('cookies',               'Third-party cookies'),
    ('canvas_fingerprinters', 'Canvas fingerprinting'),
    ('session_recorders',     'Session recording'),
    ('key_logging',           'Key logging'),
    ('fb_pixel_events',       'Facebook Pixel'),
    ('ga',                    'Google Analytics (remarketing)'),
    ('tiktok_pixel_events',   'TikTok Pixel'),
    ('twitter_pixel_events',  'X Pixel'),
]

out = []
for ct, label in CATEGORIES:
    jan = df.get(f'jan_{ct}')
    apr = df[f'apr_{ct}']
    # A card can be absent from a payload because the scan did not run that
    # test, not because the behaviour was absent. Filling those with False
    # deflated January and inflated every positive delta. Compare only the
    # domains where both scans actually reported on this card.
    comparable = jan is not None and jan.notna().any()
    if comparable:
        both = jan.notna() & apr.notna()
        n_both, n_total = int(both.sum()), len(apr)
        out.append({
            'category': label,
            'jan_prev_%': jan[both].mean() * 100,
            'apr_prev_%': apr[both].mean() * 100,
            'delta_pp': (apr[both].mean() - jan[both].mean()) * 100,
            'n_compared': n_both,
            'n_dropped': n_total - n_both,
        })
    else:
        # Card absent from the January payload version entirely (TikTok/X
        # pixels post-date it), so April is reported alone.
        out.append({
            'category': label,
            'jan_prev_%': None,
            'apr_prev_%': apr.fillna(False).mean() * 100,
            'delta_pp': None,
            'n_compared': int(apr.notna().sum()),
            'n_dropped': int(apr.isna().sum()),
        })

stability = pd.DataFrame(out)
print(f"Domains rescanned in April: {len(df):,}")
print("Per card, domains reporting at BOTH dates (the rest are dropped, not "
      "treated as absent):")
for r in stability.itertuples():
    print(f"  {r.category:32s} compared {r.n_compared:>4,}  dropped {r.n_dropped:>4,}")
stability


Domains rescanned in April: 500
Per card, domains reporting at BOTH dates (the rest are dropped, not treated as absent):
  Ad trackers                      compared  500  dropped    0
  Third-party cookies              compared  500  dropped    0
  Canvas fingerprinting            compared  500  dropped    0
  Session recording                compared  500  dropped    0
  Key logging                      compared  500  dropped    0
  Facebook Pixel                   compared  500  dropped    0
  Google Analytics (remarketing)   compared  500  dropped    0
  TikTok Pixel                     compared  500  dropped    0
  X Pixel                          compared  500  dropped    0


,category,jan_prev_%,apr_prev_%,delta_pp,n_compared,n_dropped
0,Ad trackers,67.4,67.2,-0.2,500,0
1,Third-party cookies,57.6,53.6,-4.0,500,0
2,Canvas fingerprinting,12.4,20.6,8.2,500,0
3,Session recording,10.0,9.2,-0.8,500,0
4,Key logging,3.8,4.2,0.4,500,0
5,Facebook Pixel,21.6,16.6,-5.0,500,0
6,Google Analytics (remarketing),2.8,23.8,21.0,500,0
7,TikTok Pixel,NaN,3.8,NaN,500,0
8,X Pixel,NaN,6.6,NaN,500,0


### Coverage of the 500 rescanned domains

In [5]:
yg = pd.read_csv(YG_DOMAIN).clean_names()
target_domains = set(df['domain'])
sub = yg.query('private_domain in @target_domains')
sub.shape

(37875, 4)

In [6]:
coverage = pd.Series({
    'n_domains':            len(target_domains),
    'panelists_touched':    sub['caseid'].nunique(),
    'pct_panelists':        100 * sub['caseid'].nunique() / yg['caseid'].nunique(),
    'pct_total_visits':     100 * sub['visits'].sum() / yg['visits'].sum(),
    'pct_total_duration':   100 * sub['duration'].sum() / yg['duration'].sum(),
}).round(2)
coverage

n_domains              500.00
panelists_touched     1129.00
pct_panelists           99.56
pct_total_visits        58.91
pct_total_duration      59.32
dtype: float64

In [7]:
# Per-panelist share: how much of an individual's browsing happens on these 500 domains?# 
total  = yg.groupby('caseid')[['visits', 'duration']].sum()
in_tgt = sub.groupby('caseid')[['visits', 'duration']].sum()

shares = (in_tgt / total).rename(
    columns={'visits': 'visit_share', 'duration': 'duration_share'}
)
shares.describe().round(3)

,visit_share,duration_share
count,1129.000,1129.000
mean,0.543,0.542
std,0.196,0.240
min,0.002,0.000
25%,0.398,0.365
50%,0.559,0.566
75%,0.689,0.727
max,1.000,1.000
